In [1]:
import os
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [14]:
from sparkSession import spark
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from silver_schema_def import schema_silver_events

In [15]:
dataframe = spark.read.json('/Users/ayusman/thisWorks/DE/lakehouse_analytics/data/streaming')

In [16]:
READ_PATH_CLICKSTREAM = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_clickstream'
READ_PATH_USER = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_user'

In [17]:
dataframe_clickstream = spark.read.format('delta').load(READ_PATH_CLICKSTREAM)
dataframe_user = spark.read.format('delta').load(READ_PATH_USER)

In [18]:
dataframe_clickstream.select('user_id').show(2)

+-------+
|user_id|
+-------+
|user_75|
|user_97|
+-------+
only showing top 2 rows


In [19]:
## trim everything everywhere
dataframe_clickstream = dataframe_clickstream.withColumns({c: lower(trim(col(c))) for c in dataframe_clickstream.columns})

In [20]:
dataframe_clickstream.printSchema()

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [60]:
dataframe_clickstream.select('quantity').distinct().show()
## since we have negative and null values in the quantity, we will first be removing the negatives and later dropping nulls according to eventtype
dataframe_clickstream = dataframe_clickstream.filter((F.col('quantity')>=0) | (F.col('quantity').isNull()))
dataframe_clickstream.select('quantity').distinct().show()

+--------+
|quantity|
+--------+
|       0|
|       5|
|       1|
|       3|
|       2|
|       4|
|    NULL|
+--------+

+--------+
|quantity|
+--------+
|       0|
|       5|
|       1|
|       3|
|       2|
|       4|
|    NULL|
+--------+



In [59]:
dataframe_clickstream.select(col('event_time').isNull()).count()
dataframe_clickstream.select(min('event_time'), max('event_time')).show(truncate=False)
dataframe_clickstream.filter(col('event_time')<='1970').show()
'2026-09-26t21:10:09.101217+00:00'
dataframe_clickstream = dataframe_clickstream.withColumn('event_time', upper(col('event_time')))
dataframe_clickstream.select('event_time').show(5, truncate=False)

+--------------------------------+--------------------------------+
|min(event_time)                 |max(event_time)                 |
+--------------------------------+--------------------------------+
|2026-09-03T15:10:37.101217+00:00|2027-09-02T19:07:25.101217+00:00|
+--------------------------------+--------------------------------+

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+

+--------------------------------+
|event_time                      |
+--------------------------------+
|2026-09-19T21:03:10.101217+00:00|
|2026-

In [1]:
dataframe_clickstream.select('event_time').show(4, truncate=False)

NameError: name 'dataframe_clickstream' is not defined

In [47]:
## chaned the datatype for event_time and quantity columns
dataframe_clickstream = dataframe_clickstream\
    .withColumn('quantity', to_number(col('quantity'), lit('99')))
#.withColumn('event_time', to_timestamp(col('event_time')))\
dataframe_clickstream.show(2)

+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|    category|country| device|            event_id|          event_time|event_type| price|product_id|quantity| session_id|user_id|        ingestion_ts|        ip_file_name|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|prescription|germany|desktop|3ff90190-f6e2-48c...|2026-09-19T21:03:...|  checkout|137.73|   prod_14|       4|session_407|user_75|2026-10-03 21:29:...|file:///users/ayu...|
|    cold_flu| france| mobile|1b8ba84c-525c-46f...|2026-09-09T17:15:...| page_view| 96.08|   prod_19|    NULL|session_430|user_97|2026-10-03 21:29:...|file:///users/ayu...|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+-----

In [48]:
dataframe_clickstream.printSchema()
# event_id, event_time, session_id, user_id, product_id for event_type purchase

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: decimal(2,0) (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [52]:
## deduping by event_id
df_cs_dedup = dataframe_clickstream.dropDuplicates(['event_id'])
df_cs_null_drop.select('event_type', 'quantity').filter(col('event_type') == 'add_to_cart').distinct().show(40)

+-----------+--------+
| event_type|quantity|
+-----------+--------+
|add_to_cart|       5|
|add_to_cart|       3|
|add_to_cart|       0|
|add_to_cart|       4|
|add_to_cart|       2|
|add_to_cart|       1|
+-----------+--------+



In [53]:
## dropping null user_id rows, null product_id, quantity rows for event_type purchase, add_to_cart and checkout
df_cs_null_drop = df_cs_dedup.dropna(how='any', subset=['user_id'])
df_cs_null_drop = df_cs_null_drop.filter(~(((col('product_id').isNull()) | col('quantity').isNull()) & (col('event_type').isin('add_to_cart', 'purchase','checkout'))))
df_cs_null_drop.select('event_type', 'product_id', 'quantity').filter((col('event_type').isin('purchase','add_to_cart', 'checkout')) & ((col('product_id').isNull()) | col('quantity').isNull())).distinct().show()

+----------+----------+--------+
|event_type|product_id|quantity|
+----------+----------+--------+
+----------+----------+--------+



In [51]:
## filling with 'unknown' for device / country / category
df_cs_null_drop = df_cs_null_drop.fillna('unknown', subset=['device', 'country', 'category'])
df_cs_null_drop.filter((col('device').isNull()) | (col('country').isNull()) | (col('category').isNull())).show()

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+



In [2]:
df_cs_null_drop.select('event_time').show(4, truncate=False)

NameError: name 'df_cs_null_drop' is not defined

In [57]:
## removing future date events
df_cs_fd = df_cs_null_drop.filter(~(col('event_time') > current_timestamp()))
df_cs_fd.select(max('event_time')).show(2)

+--------------------+
|     max(event_time)|
+--------------------+
|2026-10-03T12:00:...|
+--------------------+



In [ ]:
## making the strings consistant>> trim and lower

In [61]:
## checking for nulls across each column
print("Null counts across columns")
for colss in dataframe.columns:
    df_null = dataframe.filter(col(colss).isNull()).count()
    print(f"{colss}: {df_null}")

Null counts across columns
category: 10
country: 9
device: 13
event_id: 0
event_time: 0
event_type: 0
price: 0
product_id: 13
quantity: 888
session_id: 0
user_id: 3


In [ ]:
df_cs_fd.printSchema()

In [ ]:
'''df_cs_fd = df_cs_fd.withColumn('event_id', coalesce(col('event_id'), lit('unknown'))).withColumn('user_id', coalesce(col('user_id'), lit('unknown')))\
        .withColumn('session_id', coalesce(col('session_id'), lit('unknown'))).withColumn('event_time', coalesce(col('event_time'), lit('unknown')))\
        .withColumn('device', col('device'))
df_cs_fd.printSchema()'''